# Data ingestion with CREATE TABLE AS and COPY INTO

In [0]:
%sql
LIST '/Volumes/workspace/dbacademy/school_data/DE-Associate-Book/datasets/school/students-json/'

In [0]:
%sql
SELECT * 
FROM json.`/Volumes/workspace/dbacademy/school_data/DE-Associate-Book/datasets/school/students-json/`
LIMIT 5;

## 1 Batch data ingestion
### 1.1 Using Spark SQL - CTAS with read_file()

In [0]:
%sql
DROP TABLE IF EXISTS dbacademy.students_json;

CREATE TABLE dbacademy.students_json AS
SELECT *
FROM read_files('/Volumes/workspace/dbacademy/school_data/DE-Associate-Book/datasets/school/students-json/', format=>'json');

SELECT * FROM students_json LIMIT 5;

In [0]:
%sql
DESCRIBE TABLE EXTENDED dbacademy.students_json

### 1.2 Using PySpark - DataFrame with spark.read.format()

In [0]:
# Read data from file in volume to Spark DataFrame
df = (spark.read
      .format("csv")
      .option("header", "true")
      .option("sep", ";")
      .load("/Volumes/workspace/dbacademy/school_data/DE-Associate-Book/datasets/school/courses-csv/"))

# Write data from Spark DataFrame to delta table
df.write.format("delta").mode("overwrite").saveAsTable("dbacademy.courses_csv")

# Read and View the table
df_courses = spark.table("dbacademy.courses_csv")
display(df_courses)


## 2. Incremental data ingestion with COPY INTO
### 2.1 Ingesting file with COPY INTO
> Example 1: Common Schema mistmatch error

In [0]:
%sql
--Drop table if exists
DROP TABLE IF EXISTS dbacademy.courses_csv_schema_mismatch;

-- Create table with missing price column
CREATE TABLE dbacademy.courses_csv_schema_mismatch
(course_id STRING, 
title STRING, 
instructor STRING, 
category STRING);

--Copy data from file into table incrementally 
COPY INTO dbacademy.courses_csv_schema_mismatch
FROM '/Volumes/workspace/dbacademy/school_data/DE-Associate-Book/datasets/school/courses-csv/'
FILEFORMAT = CSV
FORMAT_OPTIONS (
    'header' = 'true',
    'delimiter' = ';'
    )

In [0]:
%sql
COPY INTO dbacademy.courses_csv_schema_mismatch
FROM '/Volumes/workspace/dbacademy/school_data/DE-Associate-Book/datasets/school/courses-csv/'
FILEFORMAT = CSV
FORMAT_OPTIONS (
    'header' = 'true',
    'delimiter' = ';'
    )
COPY_OPTIONS('mergeSchema' = 'true') ; --merge schema of each file

SELECT * FROM dbacademy.courses_csv_schema_mismatch LIMIT 10;

> Example 2: Preempitively Handling Schema Evolution

In [0]:
%sql
-- Drop table if exists
DROP TABLE IF EXISTS dbacademy.students_json_no_schema;

--Create an empty table without specified schema
CREATE TABLE dbacademy.students_json_no_schema;

--Copy data from file into table incrementally 
COPY INTO dbacademy.students_json_no_schema
FROM '/Volumes/workspace/dbacademy/school_data/DE-Associate-Book/datasets/school/students-json/'
FILEFORMAT = JSON
COPY_OPTIONS('mergeSchema' = 'true');  --enable Schema Evolution for table

SELECT * FROM dbacademy.students_json_no_schema LIMIT 5;
    


> Example 3: Idempotency

In [0]:
%sql
--Copy data from file into table incrementally 
-- Since the same file, no new records, no rows add into the table
COPY INTO dbacademy.students_json_no_schema
FROM '/Volumes/workspace/dbacademy/school_data/DE-Associate-Book/datasets/school/students-json/'
FILEFORMAT = JSON
COPY_OPTIONS('mergeSchema' = 'true');  --enable Schema Evolution for table

In [0]:
%sql
DROP TABLE IF EXISTS dbacademy.students_json_schema_mismatch;
DROP TABLE IF EXISTS dbacademy.students_json_no_schema;
DROP TABLE IF EXISTS dbacademy.courses_csv;
DROP TABLE IF EXISTS dbacademy.students_json;
DROP TABLE IF EXISTS dbacademy.courses_csv_schema_mismatch;
